In [2]:
# pobieramy wszystko co zwykle plus pydantica do base model, który pozwala na evaliuacje czy dany tekst ma sens

from pydantic import BaseModel 
from dotenv import load_dotenv
from pypdf import PdfReader 
from openai import OpenAI


In [3]:

load_dotenv(override=True)

myOpenAI = OpenAI()

In [4]:
# pobieramy plik pdf

pdfFile = PdfReader('pzpPDF.pdf')
pdfText = ''
for page in pdfFile.pages:
    text = page.extract_text()
    # robimy sprawdzenie czy string trafił do danego text
    if (text):
        pdfText += text

# pobieramy plik txt

with open('pzp.txt','r',encoding= 'utf-8') as f:
    txtText = f.read()


    print(pdfFile,pdfText)

<pypdf._reader.PdfReader object at 0x0000012121391A60> Strona 1
Podstawy zarz■dzania projektami
dr Tomasz Kopczy■ski
Strona 2
Wyzwania
PM
Zarz■dzanie
sytuacjami
kryzysowymi
Zarz■dzanie
zespo■ami
rozproszonymi i
zró■nicowanymi
Zwinne i
elastyczne
zespo■y
Zarz■dzanie
kompetencjami
i emocjami
Strona 3
 PROJEKT– tymczasowe przedsi■wzi■cie, maj■ce na celu
stworzenie szczególnej warto■ci (produkt, efekt), gdzie
tymczasowo■■ oznacza, ■e przedsi■wzi■cie ma okre■lony
pocz■tek i koniec.
 ZARZ■DZANIE PROJEKTEM – dzia■anie, w trakcie którego
osoba kieruj■ca projektem (kierownik projektu) przeprowadza
wraz z zespo■em planowanie oraz realizuje i monitoruje projekt
stosuj■c odpowiednie metody i dokumenty, aby osi■gn■■
wyznaczony cel w okre■lonym czasie, zakresie, kosztach i
osi■gaj■c odpowiedni■ jako■■ ko■cow■ przedsi■wzi■cia.
Strona 4
4 edycja (2009)
460 stron
Kluczowe procesy,
produkty, narz■dzi
i techniki
6 edycja (2017)
760 stron
Elementy
podej■cia
zwinnego
5 edycja (2013)
590 stron
Nowy obszar

In [5]:
# system prompt  

system_prompt = f"""
Jesteś asystentem eksperckim wspierającym użytkownika w zrozumieniu zarządzania projektami.

Odpowiadasz na pytania na podstawie dostarczonych materiałów, szczególnie dotyczących:
- definicji projektu i zarządzania projektem
- etapów projektu (inicjowanie, planowanie, realizacja, zamknięcie)
- trójkąta projektowego (zakres, czas, budżet, jakość)
- ryzyka, interesariuszy i ról w projekcie

Odpowiadaj jasno, konkretnie i w sposób uporządkowany.
Jeśli to możliwe, podawaj przykłady praktyczne.

Jeśli nie znasz odpowiedzi — powiedz to wprost.

---

## Materiały:

### Dokument:
{txtText}

### PDF:
{pdfText}

---

Na podstawie tego kontekstu odpowiadaj na pytania użytkownika.
"""

In [6]:
# tworzymy obiekt za pomocą słowa class , będzie on słuzył do weryfikacji odpowiedzi modelu w nawiasie jako argument przekazujemy Basemodel czyli
#klase z biblioteki pydantic odpowiadajdzą za prace na danych, dzieki niej nasz obiekt będzie weryikował odpowiedz przez konkretne dane jakie wskazemy dalej
# będzie walidował / sprawdzał czy odpowiedz jest dobra poprzez zdefiniowane pola
# SAM W SOBIE NIC NIE SPRAWDZA, DOPIERO POTEM JAK NAPISZEMY MODEL KTÓRY FAKTUCZNIE SPRAWDZA I DA NAM ODPOWIEDZ TA ODPOWIEDZ PRZEJDZIE PRZEZ TA KLASE
# I ODPOWIEDZ Z MODELU BEDZIE UŁOZONA ZGODNIE Z TYM OBIEKTEM I JEGO POLAMI DOSTANE INFO CZY ODPOWIEDZ JEST TRUE CZY FALSE DZIEKI POLU IS_ACCEPT
# I KRÓTKI FEEDBACK DLACZEGO JEST DOBRA ALBO ZŁA // NIC NIE SPRAWDZA BO NIE MA W NIEJ MODELU OPENAI ANI NIC TAKIEGO, DOPIERO PRAWDZIWA ODPOWIEDZ PRZEJDZIE PRZEZ MODEL
# ZEBY BYŁA SCHLUDNIE PODANA USEROWI

class Evaluation(BaseModel):
    is_accept: bool
    feedback: str



In [7]:
# prompt dla evaluatora

evaluator_system_prompt = f"""
Jesteś ewaluatorem odpowiedzi AI.

Twoim zadaniem jest ocenić, czy odpowiedź jest zgodna z dostarczonymi materiałami dotyczącymi zarządzania projektami.

Weź pod uwagę:
- poprawność merytoryczną
- zgodność z koncepcjami (etapy projektu, trójkąt projektowy, ryzyko, interesariusze, role)
- czy odpowiedź jest jasna i użyteczna

Jeśli odpowiedź zawiera błędy, nieścisłości lub wychodzi poza kontekst — oznacz ją jako nieakceptowalną.

---

## Materiały:

### Dokument:
{txtText}

### PDF:
{pdfText}

---

Zwróć WYŁĄCZNIE JSON w formacie:
 is_acceptable (bool), feedback (string)
"""

In [8]:
# teraz piszemy funkcje którra przygotowuje wiadomsc dla evaluatora czyli 2 modelu ai który zgodnie z promptami sprawdzi zasadnosc odpowiedzi
# przesłanej przez zwykły model funkcja w argumentach przyjmuje reply - odpowiedz modelu który generuje odpowiedzi, message, czyli moją wiadomosc do modelu
# generującego odpowiedzi oraz history czyli tablice gdzie appendujemy reply i message, czyli odpowiedzi i pytania 
#evaluator (model AI) musi dostać pełny kontekst, żeby dobrze ocenić odpowiedź/ FUNKCJA JEST WŁACZANA PÓZNIEJ W KODZIE
def evaluator_user_prompt(reply,message,history):
    # w ciele funkcji budujemy prompt, który trafi do evaluatora (czyli drugiego modelu AI oceniającego odpowiedź)
    user_prompt = f"Oto przebieg rozmowy:\n{history}\n\n"
    user_prompt += f"Pytanie użytkownika:\n{message}\n\n"
    user_prompt += f"Odpowiedź agenta:\n{reply}\n\n"
    user_prompt += "Oceń tę odpowiedź."
    return user_prompt

    #return zwraca wynik funkcji, czyli user_prompt, gdy funkcja zostanie wywołana i  przypisuje wymik czyli nasz prompt do zmiennej
    # ta funkcja zwraca reply czyli odpowiedz zwykłego modelu, message usera i tablice z historią rozmowy

In [9]:
# budujemy funkcje która otrzymuje w argumencie, reply czylu odpowiedz modelu kóry odpowiada na pytania, message usera do modelu oraz tablice history
#łaczy dane czyli wiadomosci w tekst przygotowuje ją dla evaluotra czyli 2 modelu ai kóry analizuje odpowiedz modelu
# I TA FUNKCJA JEST TEZ EVALUTOREM, W CIELE FUNKCJI DALEJ TWORZYMY I URUCHAIAMY MODEL AI KTÓRY MA ZA ZADANIE SPRAWDZIC ODPOWEDZ 1 MODELU NA PODSTAWIE
# WYTYCZNYCH Z EVALUTOR_SYSTEM_PROMPT = PISANY RECZNIE I WYNIK FUNKCJU EVALUATOR USER PROMPT, GDZIE TRAFIA REPLY MESSAGE HISTORY 
# TA FUNKCJA URUCHAMIA TEZ def evaluator_user_prompt(reply,message,history): i POTEM URUCHAMIA MODEL AI KTÓRY DZIEKI RETURNOWI Z FUNKCJU MA WSZYSTKIE DANE
# Z 1 MODELU  
# wynik tej funkcji trafia tez do class Evaluation(BaseModel): który porządkuje dane i daje nam odpowiedz evaluatora uporządkowaną

def evaluator(reply,message,history) -> Evaluation:

    # wpierw budujemy message który zawiera message, prompty 1 to nasz promt systemowy 2 to wynik funkcji evaluator user prompt iktóra wywołujemy
    messages = [{'role':'system','content': evaluator_system_prompt},
                {'role':'user','content': evaluator_user_prompt(reply,message,history)}
    ]  

    # teraz budujemy odpowiedz modelu ale troche inaczej, poniewaz nasza odpowiedz ma nie byc stringiem tylko ma trafic do klasy (obiektu) Evaluation
    # zamiast create uzywamy parse, który zamieni ją na obiekt zeby evaluation mógl na  niej działać
    # dzieki parse zmiast create model zwróci odpowiedz w obiekcie evaluation

    response = myOpenAI.beta.chat.completions.parse(
        model='gpt-4.1-mini',
        messages=messages,
        response_format=Evaluation
    )

    # response_format=Evaluation mówi modelowi: zwróć odpowiedź w formacie tej klasy (Evaluation)” dostaniemy dizeki temu ine string a 
    #  evaluation.is_acceptable evaluation.feedback

    # zwracamy response
    return response.choices[0].message.parsed


In [10]:
# piszemy funkcje rerun, która ponownie uruchmomi 1 model jesli model evalautor który wysle odpowiedz do klasy Evalaution uznają że model odpowiedział
# niezgodnie z wytycznymi usera, funkcja przyjmuje reply czyli odpowiedz 1 modelu, message czyli wiadomosc zdefinianwną przez usera, array history
# ORAZ informacje string feedback z class Evaluation FUNKCJA JEST URUCHAMIANA POTEN W WARNUKU IF JESLI EVALAUTOR (2 MODEL) UZNA ŻE NEIZBEDE JEST JESZE RAZ
# WYWOŁAC FUNKCJE # zawiera tez model odpowiedzi wiec nie wywołuje ponownie 1 modelu ale zbiera dane z 1 modelu z 2 i mając je sam tworzy poprawna odpowiedź


def rerun(reply,message,history,feedback):
    # TWORZYMY NOWY SYSTEM PROPMT POWIĘSZKONY O FEEDBACK Z EBALUATORA // przekazujemy poprzedni system prompt i powiększamy go o evaluatora
    updated_system_prompt = system_prompt + "\n\n## Poprzednia odpowiedź była błędna\n"
    updated_system_prompt += f"Twoja odpowiedź:\n{reply}\n\n"
    updated_system_prompt += f"Feedback:\n{feedback}\n\n"

    # tworzymy listę messages dla modelu, zawieraącą info od systemu, nowy system prompt, history i wiadoomosc usera
    messages = [ {'role':'system',"content": updated_system_prompt} ]+ history + [{'role':'user','content':message}]
    
    

    # teraz tworzymy model 

    response = myOpenAI.chat.completions.create(

        model='gpt-4.1-mini',
        messages=messages
    )

    return response.choices[0].message.content
